In [24]:
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import h3
import libpysal
from libpysal.weights import KNN, W
from esda.moran import Moran
from spreg import ML_Error

In [25]:
TABLE_DIR = "../output/tables"
os.makedirs(TABLE_DIR, exist_ok=True)

# Refit the OLS models

In [26]:
data_path = "../output/data/merged_analysis_data_standardized.pkl"
df = pd.read_pickle(data_path)

industry_codes = [
    "A","B","C","D","E","F","G","H","I","J","K","L","M","N","O","P","Q","R","S"
]
industry_bool_cols = [f"industry_{c}" for c in industry_codes]

df["industry_fe"] = df[industry_bool_cols].idxmax(axis=1).str.replace("industry_", "")
df["industry_fe"] = pd.Categorical(df["industry_fe"])

formula_rhs = """
    loc_work_gaussian_opening
    + productivity
    + n_companies
    + entropy
    + C(industry_fe, Treatment("G"))
    + distance_to_deak
    + work_cluster_1
    + work_cluster_0
    + arpu_high_ratio_open
    + income_entr_open
    + sex_fem_ratio_open
"""

formula_inter_fem = """
work_abs_diff ~
    income_entr_open
    + sex_fem_ratio_open
    + arpu_high_ratio_open
    + distance_to_deak:sex_fem_ratio_open
    + loc_work_gaussian_opening
    + productivity
    + n_companies
    + entropy
    + C(industry_fe, Treatment("G"))
    + distance_to_deak
    + work_cluster_1
    + work_cluster_0
"""

formula_inter_inc = """
work_abs_diff ~
    income_entr_open
    + sex_fem_ratio_open
    + arpu_high_ratio_open
    + distance_to_deak:income_entr_open
    + loc_work_gaussian_opening
    + productivity
    + n_companies
    + entropy
    + C(industry_fe, Treatment("G"))
    + distance_to_deak
    + work_cluster_1
    + work_cluster_0
"""

In [27]:
# Build one consistent modeling sample
needed_cols = [
    "work_abs_diff", "raster_id", "loc_work_gaussian_opening", "productivity", "n_companies", "entropy",
    "industry_fe", "distance_to_deak", "work_cluster_1", "work_cluster_0",
    "arpu_high_ratio_open", "income_entr_open", "sex_fem_ratio_open",
]
df_model = df.dropna(subset=needed_cols).copy()

formula_full = "work_abs_diff ~ " + formula_rhs
model_full = smf.ols(formula_full, data=df_model).fit(cov_type="HC3")
model_inter_fem = smf.ols(formula_inter_fem, data=df_model).fit(cov_type="HC3")
model_inter_low_inc = smf.ols(formula_inter_inc, data=df_model).fit(cov_type="HC3")

# Spatial weights

In [28]:
# H3 centroids, computed once on df_model only
lat_lng = df_model["raster_id"].apply(lambda hx: h3.h3_to_geo(hx))
df_model["h3_lat"] = lat_lng.apply(lambda x: x[0])
df_model["h3_lng"] = lat_lng.apply(lambda x: x[1])
coords_model = list(zip(df_model["h3_lng"], df_model["h3_lat"]))

In [29]:
# using H3 hexagon adjacency for spatial closeness
import collections

def build_h3_contiguity_weights_adaptive(raster_ids, max_ring=5):
    """H3 adjacency weights, expanding the ring size per-hex only when needed: if a hex has
    no in-sample neighbor at ring=1, grow the search radius (ring=2, 3, ...) until one is
    found, up to max_ring. Hexes that still have no neighbor at max_ring are left as
    islands and reported separately."""
    raster_id_set = set(raster_ids)
    neighbors = {}
    rings_used = {}

    for rid in raster_ids:
        found = []
        ring = 1
        while ring <= max_ring and not found:
            candidates = h3.k_ring(rid, ring) - {rid}
            found = [n for n in candidates if n in raster_id_set]
            ring += 1
        neighbors[rid] = found
        rings_used[rid] = (ring - 1) if found else None

    return W(neighbors), rings_used

w_contiguity, rings_used = build_h3_contiguity_weights_adaptive(df_model["raster_id"], max_ring=5)

unresolved = [rid for rid, r in rings_used.items() if r is None]
ring_dist = collections.Counter(r for r in rings_used.values() if r is not None)

print(f"Adaptive H3 contiguity - components: {w_contiguity.n_components}, "
      f"islands: {len(w_contiguity.islands)}, unresolved at max_ring=5: {len(unresolved)}")
print("Ring distance needed to find a neighbor:", dict(sorted(ring_dist.items())))

Adaptive H3 contiguity - components: 367, islands: 4, unresolved at max_ring=5: 4
Ring distance needed to find a neighbor: {1: 8415, 2: 370, 3: 58, 4: 8, 5: 6}


/tmp/ipykernel_3489220/2160078903.py:23: UserWarning: The weights matrix is not fully connected: 
 There are 367 disconnected components.
 There are 4 islands with ids: 8a1e03612447fff, 8a1e03685d6ffff, 8a1e037b3b67fff, 8a1e1cb6d587fff.
  return W(neighbors), rings_used


In [30]:
if unresolved:
    print(f"Dropping {len(unresolved)} still-isolated raster_ids before SEM estimation.")
    df_model = df_model[~df_model["raster_id"].isin(unresolved)].copy()
    # rebuild w_contiguity, X, y from this filtered df_model before re-running ML_Error

w_contiguity.transform = "r"

Dropping 4 still-isolated raster_ids before SEM estimation.
('WARNING: ', '8a1e03612447fff', ' is an island (no neighbors)')
('WARNING: ', '8a1e03685d6ffff', ' is an island (no neighbors)')
('WARNING: ', '8a1e037b3b67fff', ' is an island (no neighbors)')
('WARNING: ', '8a1e1cb6d587fff', ' is an island (no neighbors)')


In [31]:
# REBUILD weights from the cleaned df_model 
w_contiguity, rings_used = build_h3_contiguity_weights_adaptive(df_model["raster_id"], max_ring=5)

w_contiguity.transform = "r"

/tmp/ipykernel_3489220/2160078903.py:23: UserWarning: The weights matrix is not fully connected: 
 There are 363 disconnected components.
  return W(neighbors), rings_used


# Moran's I: H3 contiguity weights

In [32]:
# Refit on the final df_model, so residuals align with w_contiguity
model_full = smf.ols(formula_full, data=df_model).fit(cov_type="HC3")
model_inter_fem = smf.ols(formula_inter_fem, data=df_model).fit(cov_type="HC3")
model_inter_low_inc = smf.ols(formula_inter_inc, data=df_model).fit(cov_type="HC3")

models = {
    "Baseline OLS":        model_full,
    "Interaction: Female": model_inter_fem,
    "Interaction: Income": model_inter_low_inc,
}

In [33]:
def compute_moran_table(models, w, w_name):
    results = {}
    for name, model in models.items():
        mi = Moran(model.resid.values, w)
        results[name] = {
            "Moran's I":  round(mi.I, 4),
            "Expected I": round(mi.EI, 4),
            "z-score":    round(mi.z_norm, 3),
            "p-value":    round(mi.p_norm, 4),
        }
    moran_df = pd.DataFrame(results).T
    print(f"\n--- Moran's I ({w_name}) ---")
    print(moran_df.to_string())
    return moran_df

moran_contiguity = compute_moran_table(models, w_contiguity, "H3 contiguity, ring=1")


--- Moran's I (H3 contiguity, ring=1) ---
                     Moran's I  Expected I  z-score  p-value
Baseline OLS            0.6819     -0.0001   72.928      0.0
Interaction: Female     0.6788     -0.0001   72.600      0.0
Interaction: Income     0.6817     -0.0001   72.907      0.0


# Spatial Error Models (SEM)

In [34]:
df_model = df_model.set_index("raster_id").loc[w_contiguity.id_order].reset_index()

X_vars = ["loc_work_gaussian_opening", "productivity", "n_companies", "entropy",
          "distance_to_deak", "work_cluster_1", "work_cluster_0",
          "arpu_high_ratio_open", "income_entr_open", "sex_fem_ratio_open"]

y = df_model["work_abs_diff"].values.reshape(-1, 1)

industry_dummies = (
    pd.get_dummies(df_model["industry_fe"], prefix="ind", dtype=int)
      .drop(columns=["ind_G"])
)
X = pd.concat([df_model[X_vars], industry_dummies], axis=1).astype(float).values

In [35]:
serr = ML_Error(y, X, w_contiguity,
                name_y="work_abs_diff",
                name_x=X_vars + list(industry_dummies.columns))
print(serr.summary)

/mnt/common-ssd/zadorzsofi/telekom/.venv/lib/python3.10/site-packages/spreg/ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(
/mnt/common-ssd/zadorzsofi/telekom/.venv/lib/python3.10/site-packages/spreg/ml_error.py:563: RuntimeWarning: divide by zero encountered in log
  jacob = np.log(np.linalg.det(a))


ML_Error
REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ML SPATIAL ERROR (METHOD = full)
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :work_abs_diff                Number of Observations:        8857
Mean dependent var  :     -0.0000                Number of Variables   :          29
S.D. dependent var  :      1.0003                Degrees of Freedom    :        8828
Pseudo R-squared    :      0.3220
Log likelihood      :  -7086.4002
Sigma-square ML     :      0.2452                Akaike info criterion :   14230.800
S.E of regression   :      0.4951                Schwarz criterion     :   14436.380

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
----------------------------------------------------------------------------------

In [36]:
# Interaction: Female ratio x distance
df_model["interact_fem_dist"] = df_model["sex_fem_ratio_open"] * df_model["distance_to_deak"]
df_model["interact_inc_dist"] = df_model["income_entr_open"] * df_model["distance_to_deak"]

X_vars_fem = X_vars + ["interact_fem_dist"]
X_fem = pd.concat([df_model[X_vars_fem], industry_dummies], axis=1).astype(float).values

sem_fem = ML_Error(y, X_fem, w_contiguity,
                   name_y="work_abs_diff",
                   name_x=X_vars_fem + list(industry_dummies.columns))
print(sem_fem.summary)

/mnt/common-ssd/zadorzsofi/telekom/.venv/lib/python3.10/site-packages/spreg/ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(
/mnt/common-ssd/zadorzsofi/telekom/.venv/lib/python3.10/site-packages/spreg/ml_error.py:563: RuntimeWarning: divide by zero encountered in log
  jacob = np.log(np.linalg.det(a))


ML_Error
REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ML SPATIAL ERROR (METHOD = full)
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :work_abs_diff                Number of Observations:        8857
Mean dependent var  :     -0.0000                Number of Variables   :          30
S.D. dependent var  :      1.0003                Degrees of Freedom    :        8827
Pseudo R-squared    :      0.3307
Log likelihood      :  -7065.6825
Sigma-square ML     :      0.2440                Akaike info criterion :   14191.365
S.E of regression   :      0.4940                Schwarz criterion     :   14404.034

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
----------------------------------------------------------------------------------

In [37]:
# Interaction: Income entropy x distance
X_vars_inc = X_vars + ["interact_inc_dist"]
X_inc = pd.concat([df_model[X_vars_inc], industry_dummies], axis=1).astype(float).values

sem_inc = ML_Error(y, X_inc, w_contiguity,
                   name_y="work_abs_diff",
                   name_x=X_vars_inc + list(industry_dummies.columns))
print(sem_inc.summary)

/mnt/common-ssd/zadorzsofi/telekom/.venv/lib/python3.10/site-packages/spreg/ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(
/mnt/common-ssd/zadorzsofi/telekom/.venv/lib/python3.10/site-packages/spreg/ml_error.py:563: RuntimeWarning: divide by zero encountered in log
  jacob = np.log(np.linalg.det(a))


ML_Error
REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ML SPATIAL ERROR (METHOD = full)
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :work_abs_diff                Number of Observations:        8857
Mean dependent var  :     -0.0000                Number of Variables   :          30
S.D. dependent var  :      1.0003                Degrees of Freedom    :        8827
Pseudo R-squared    :      0.3282
Log likelihood      :  -7059.0092
Sigma-square ML     :      0.2437                Akaike info criterion :   14178.018
S.E of regression   :      0.4936                Schwarz criterion     :   14390.687

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
----------------------------------------------------------------------------------

# LaTeX regression table

In [38]:
latex_label_map_spreg = {
    "CONSTANT": "Constant",
    "loc_work_gaussian_opening": "Baseline work activity",
    "productivity": "Productivity",
    "n_companies": "Number of companies",
    "entropy": "Industry entropy",
    "distance_to_deak": "Distance to centre",
    "work_cluster_1": "Day Shift cluster",
    "work_cluster_0": "Mixed Shift cluster",
    "arpu_high_ratio_open": "High income ratio",
    "income_entr_open": "Income entropy",
    "sex_fem_ratio_open": "Female ratio",
    "interact_fem_dist": "Distance x Female ratio",
    "interact_inc_dist": "Distance x Income entropy",
    "lambda": r"$\lambda$",
}

In [39]:
def spreg_results_to_df(model):
    """Extract a tidy coefficient table from a spreg ML_Error result object, deriving
    variable names from the actual coefficient array length rather than assuming whether
    name_x already includes CONSTANT/lambda (this varies by spreg version)."""
    n_betas = len(model.betas)
    name_x = list(model.name_x)

    if len(name_x) == n_betas:
        names = name_x
    elif len(name_x) == n_betas - 1:
        names = name_x + ["lambda"]
    elif len(name_x) == n_betas - 2:
        names = ["CONSTANT"] + name_x + ["lambda"]
    else:
        raise ValueError(
            f"Can't reconcile name_x length ({len(name_x)}) with betas length ({n_betas}) "
            f"for this model - inspect model.name_x and model.betas directly."
        )

    coefs = model.betas.flatten()
    se = model.std_err.flatten()
    z_stats = [z for z, p in model.z_stat]
    p_vals = [p for z, p in model.z_stat]

    return pd.DataFrame({"variable": names, "coef": coefs, "std_err": se, "z": z_stats, "p": p_vals})

In [43]:
def save_spreg_latex_table(models, model_names, filename_stem, directory=TABLE_DIR, label_map=None,
                            fe_prefix="ind_", fe_label="Industry FE"):
    """Build a side-by-side LaTeX regression table from a list of spreg ML_Error results.
    Fixed-effect dummies are suppressed and summarized as a single Yes/No indicator row."""
    label_map = label_map or {}

    def fmt_coef(coef, p):
        stars = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""
        return f"{coef:.3f}{stars}"

    tidy_list = [spreg_results_to_df(m).set_index("variable") for m in models]

    # separate FE dummies
    all_vars = []
    for t in tidy_list:
        for v in t.index:
            if v not in all_vars and not v.startswith(fe_prefix):
                all_vars.append(v)

    # did each model include any FE dummy?
    fe_included = [any(v.startswith(fe_prefix) for v in t.index) for t in tidy_list]

    lines = [
        r"\begin{tabular}{l" + "c" * len(models) + "}",
        r"\toprule",
        " & " + " & ".join(model_names) + r" \\",
        r"\midrule",
    ]

    for var in all_vars:
        row_label = label_map.get(var, var).replace("_", r"\_")
        coef_row = [row_label]
        se_row = [""]
        for t in tidy_list:
            if var in t.index:
                coef_row.append(fmt_coef(t.loc[var, "coef"], t.loc[var, "p"]))
                se_row.append(f"({t.loc[var, 'std_err']:.3f})")
            else:
                coef_row.append("")
                se_row.append("")
        lines.append(" & ".join(coef_row) + r" \\")
        lines.append(" & ".join(se_row) + r" \\")

    lines.append(r"\midrule")
    lines.append(" & ".join([fe_label] + ["Yes" if inc else "No" for inc in fe_included]) + r" \\")
    lines.append(" & ".join(["N"] + [str(m.n) for m in models]) + r" \\")
    lines.append(" & ".join(["Pseudo R$^2$"] + [f"{m.pr2:.3f}" for m in models]) + r" \\")
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")

    with open(f"{directory}/{filename_stem}.tex", "w") as f:
        f.write("\n".join(lines))

In [44]:
save_spreg_latex_table(
    [serr, sem_fem, sem_inc],
    model_names=["Baseline", "Female Interact", "Income Interact"],
    filename_stem="sem_regression_table",
    label_map=latex_label_map_spreg,   
)

In [42]:
def save_spreg_latex_table(models, model_names, filename_stem, directory=TABLE_DIR, label_map=None):
    """Build a side-by-side LaTeX regression table from a list of spreg ML_Error results."""
    label_map = label_map or {}

    def fmt_coef(coef, p):
        stars = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""
        return f"{coef:.3f}{stars}"

    tidy_list = [spreg_results_to_df(m).set_index("variable") for m in models]

    all_vars = []
    for t in tidy_list:
        for v in t.index:
            if v not in all_vars:
                all_vars.append(v)

    lines = [
        r"\begin{tabular}{l" + "c" * len(models) + "}",
        r"\toprule",
        " & " + " & ".join(model_names) + r" \\",
        r"\midrule",
    ]

    for var in all_vars:
        row_label = label_map.get(var, var).replace("_", r"\_")   # <- look up label first
        coef_row = [row_label]
        se_row = [""]
        for t in tidy_list:
            if var in t.index:
                coef_row.append(fmt_coef(t.loc[var, "coef"], t.loc[var, "p"]))
                se_row.append(f"({t.loc[var, 'std_err']:.3f})")
            else:
                coef_row.append("")
                se_row.append("")
        lines.append(" & ".join(coef_row) + r" \\")
        lines.append(" & ".join(se_row) + r" \\")

    lines.append(r"\midrule")
    lines.append(" & ".join(["N"] + [str(m.n) for m in models]) + r" \\")
    lines.append(" & ".join(["Pseudo R$^2$"] + [f"{m.pr2:.3f}" for m in models]) + r" \\")
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")

    with open(f"{directory}/{filename_stem}.tex", "w") as f:
        f.write("\n".join(lines))

save_spreg_latex_table(
    [serr, sem_fem, sem_inc],
    model_names=["Baseline", "Female Interact", "Income Interact"],
    filename_stem="sem_regression_table",
    label_map=latex_label_map_spreg,   
)